# CTA RouteRanker: daily ridership model training

Standalone Google Colab notebook for a retrospective route/date ridership experiment. It validates the real CTA export, creates exact-calendar-date lags, selects a pooled regressor on validation dates, and evaluates once on held-out test dates against seasonal baselines.

**Input:** CTA Ridership - Bus Routes - Daily Totals by Route CSV with route, date, daytype (W/A/U), and rides columns. Keep the original file unchanged and note its download date. Do not use synthetic ridership.

**Interpretation:** the guide has not established when CTA publishes each observation. Because this retrospective experiment uses prior observed ridership, it is not an operational next-day forecast.

Default periods follow the intern guide: train through 2026-01-31, validate 2026-02-01 to 2026-04-30, test 2026-05-01 to 2026-07-31. Change them only if source coverage differs, and freeze them before inspecting test results.

In [ ]:
%pip -q install pandas numpy scikit-learn joblib

## 1. Load the real CSV

The notebook first downloads the official CSV into the Colab session. If that request fails, it checks Google Drive and then offers a manual upload.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from google.colab import drive, files

CSV_PATH = '/content/cta_bus_ridership.csv'
CSV_URL = 'https://data.cityofchicago.org/api/views/jyb9-n7fm/rows.csv?accessType=DOWNLOAD'
try:
    print('Downloading CTA bus ridership data from the official Chicago Data Portal...')
    urlretrieve(CSV_URL, CSV_PATH)
    print('Download complete.')
except Exception as download_error:
    print(f'Automatic download failed: {download_error}')
    drive.mount('/content/drive')
    drive_path = Path('/content/drive/MyDrive/cta_bus_ridership.csv')
    if drive_path.exists():
        CSV_PATH = str(drive_path)
    else:
        print('Upload the CTA Bus Routes - Daily Totals by Route CSV.')
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError('No CSV was uploaded and the Drive fallback was not found.')
        CSV_PATH = next(iter(uploaded))
print('CSV path:', CSV_PATH)

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd

path = Path(CSV_PATH)
if not path.exists():
    raise FileNotFoundError(f'CSV not found: {path}. Update CSV_PATH or upload the CTA export.')
source_sha256 = hashlib.sha256(path.read_bytes()).hexdigest()
print('SHA-256:', source_sha256)
raw = pd.read_csv(path, dtype='string', keep_default_na=False)
column_lookup = {str(c).strip().lower(): c for c in raw.columns}
aliases = {'date': ('date', 'service_date'), 'route': ('route', 'route_name', 'bus_route'),
           'daytype': ('daytype', 'day_type'), 'rides': ('rides', 'ridership', 'total_rides', 'boardings', 'daily_boardings')}
rename, missing = {}, []
for canonical, names in aliases.items():
    match = next((column_lookup[name] for name in names if name in column_lookup), None)
    if match is None: missing.append(canonical)
    else: rename[match] = canonical
if missing: raise ValueError(f'Missing required fields {missing}. Found: {list(raw.columns)}')
data = raw.rename(columns=rename)[['date', 'route', 'daytype', 'rides']].copy()
data['date'] = pd.to_datetime(data['date'], errors='coerce')
data['route'] = data['route'].str.strip()
data['daytype'] = data['daytype'].str.strip().str.upper()
data['rides'] = pd.to_numeric(data['rides'].str.replace(',', '', regex=False), errors='coerce')
invalid = (data['date'].isna() | data['route'].eq('') | ~data['daytype'].isin(['W', 'A', 'U']) |
           data['rides'].isna() | (data['rides'] < 0) | (data['rides'] % 1 != 0))
if invalid.any(): raise ValueError(f'{int(invalid.sum())} malformed rows found. Inspect/quarantine them; do not silently drop them.')
data['date'] = data['date'].dt.normalize()
data['rides'] = data['rides'].astype('int64')
data = data.drop_duplicates()
conflicts = data.duplicated(['route', 'date'], keep=False)
if conflicts.any():
    display(data.loc[conflicts].sort_values(['route', 'date']).head(20))
    raise ValueError(f'{int(conflicts.sum())} conflicting route/date rows found. Resolve first; never sum or average conflicts.')
data = data.sort_values(['date', 'route']).reset_index(drop=True)
print(f'{len(data):,} valid route/date observations; {data.route.nunique()} routes; {data.date.min().date()} through {data.date.max().date()}')
print('True zero-ridership rows:', int((data.rides == 0).sum()))
display(data.head())

## 2. Build leakage-aware features

Every lag joins the same route at exactly target_date minus lag_days. Missing dates stay missing; the previous available row is not treated as a seven-day lag.

In [ ]:
LAG_DAYS = (1, 7, 14, 21, 28)
SEASONAL_LAGS = (7, 14, 21, 28)
features = data.copy()
features['day_of_week'] = features.date.dt.dayofweek
features['month'] = features.date.dt.month
features['is_weekend'] = (features.day_of_week >= 5).astype('int8')
features['is_sunday_holiday'] = features.daytype.eq('U').astype('int8')
lag_source = features[['route', 'date', 'rides']]
for days in LAG_DAYS:
    lookup = lag_source.rename(columns={'date': 'source_date', 'rides': f'lag_{days}'})
    lookup['date'] = lookup.source_date + pd.to_timedelta(days, unit='D')
    features = features.merge(lookup[['route', 'date', f'lag_{days}']], on=['route', 'date'], how='left', validate='one_to_one')
seasonal_cols = [f'lag_{days}' for days in SEASONAL_LAGS]
features['seasonal_lag_count'] = features[seasonal_cols].notna().sum(axis=1)
features['seasonal_lag_mean'] = features[seasonal_cols].mean(axis=1).where(features.seasonal_lag_count >= 2)
FEATURE_COLUMNS = ['route', 'daytype', 'day_of_week', 'month', 'is_weekend', 'is_sunday_holiday',
                   'lag_1', 'lag_7', 'lag_14', 'lag_21', 'lag_28', 'seasonal_lag_mean', 'seasonal_lag_count']
print('Feature rows:', f'{len(features):,}')

## 3. Freeze chronological partitions

The guide's development window is two years of recent demand. Set alternate dates only if source coverage requires it, before checking test performance.

In [ ]:
HISTORY_START, TRAIN_END = '2024-08-01', '2026-01-31'
VALIDATION_START, VALIDATION_END = '2026-02-01', '2026-04-30'
TEST_START, TEST_END = '2026-05-01', '2026-07-31'
start, train_end = pd.Timestamp(HISTORY_START), pd.Timestamp(TRAIN_END)
val_start, val_end = pd.Timestamp(VALIDATION_START), pd.Timestamp(VALIDATION_END)
test_start, test_end = pd.Timestamp(TEST_START), pd.Timestamp(TEST_END)
if not (start <= train_end < val_start <= val_end < test_start <= test_end):
    raise ValueError('Date windows must be ordered: train, validation, test')
window = features[features.date.between(start, test_end)].copy()
train_df = window[window.date.between(start, train_end)].copy()
val_df = window[window.date.between(val_start, val_end)].copy()
test_df = window[window.date.between(test_start, test_end)].copy()
for name, part in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    print(f'{name:10s} {len(part):>8,} rows | {part.date.min().date() if len(part) else None} to {part.date.max().date() if len(part) else None}')
if any(part.empty for part in (train_df, val_df, test_df)):
    raise ValueError('A split is empty. Check source dates or adjust split dates before evaluation.')

## 4. Fit, select on validation, evaluate test once

Compare pooled Ridge regression and two tree regressors. Imputation and categorical encoding are fitted inside each training pipeline. The test period is used only after selecting a model on validation.

In [ ]:
import numpy as np
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

def metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.maximum(np.asarray(predicted, dtype=float), 0)
    total = actual.sum()
    return {'mae': float(mean_absolute_error(actual, predicted)),
            'rmse': float(mean_squared_error(actual, predicted) ** 0.5),
            'wape': float(np.abs(actual - predicted).sum() / total) if total > 0 else None}
categorical, numeric = ['route', 'daytype'], [c for c in FEATURE_COLUMNS if c not in ['route', 'daytype']]
def make_pipeline(estimator):
    preprocess = ColumnTransformer([
        ('categories', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical),
        ('numeric', SimpleImputer(strategy='median', add_indicator=True), numeric),
    ])
    return Pipeline([('features', preprocess), ('model', estimator)])
candidates = {
    'pooled_ridge': Ridge(alpha=1.0),
    'hist_gradient_boosting': HistGradientBoostingRegressor(max_iter=150, learning_rate=0.08, l2_regularization=1.0, random_state=42),
    'random_forest': RandomForestRegressor(n_estimators=250, min_samples_leaf=3, max_features=0.9, n_jobs=-1, random_state=42),
}
validation_metrics, best_name, best_mae = {}, None, float('inf')
for name, estimator in candidates.items():
    candidate = make_pipeline(estimator)
    candidate.fit(train_df[FEATURE_COLUMNS], train_df.rides)
    val_pred = np.maximum(candidate.predict(val_df[FEATURE_COLUMNS]), 0)
    validation_metrics[name] = metrics(val_df.rides, val_pred)
    if validation_metrics[name]['mae'] < best_mae:
        best_name, best_mae = name, validation_metrics[name]['mae']
print('Validation metrics:')
display(pd.DataFrame(validation_metrics).T.sort_values('mae'))
print('Selected on validation:', best_name)
fit_df = window[window.date.between(start, val_end)]
model = make_pipeline(candidates[best_name])
model.fit(fit_df[FEATURE_COLUMNS], fit_df.rides)
test_pred = np.maximum(model.predict(test_df[FEATURE_COLUMNS]), 0)
test_actual = test_df.rides.to_numpy()
print('Held-out model test metrics:', metrics(test_actual, test_pred))

## 5. Seasonal baselines and fixed-origin evaluation

The first metrics below are rolling one-step retrospective scores: later test dates can use actual rides from earlier test dates. The fixed-origin section instead issues the full May-July forecast from April 30 and recursively feeds model predictions into later lags. Its baseline fallback order is lag 7, then lag 1, then training target median. The four-week seasonal mean requires at least two same-weekday lag values.

In [ ]:
fallback = float(fit_df.rides.median())
last_week = test_df.lag_7.fillna(test_df.lag_1).fillna(fallback).to_numpy(dtype=float)
seasonal_mean = test_df[seasonal_cols].mean(axis=1).where(test_df.seasonal_lag_count >= 2)
seasonal_mean = seasonal_mean.fillna(test_df.lag_7).fillna(test_df.lag_1).fillna(fallback).to_numpy(dtype=float)
baseline_metrics = {'last_week_same_day': metrics(test_actual, last_week),
                    'past_four_same_weekdays': metrics(test_actual, seasonal_mean)}
test_results = pd.DataFrame({'route': test_df.route.to_numpy(), 'daytype': test_df.daytype.to_numpy(),
                             'actual': test_actual, 'predicted': test_pred})
route_mae = test_results.groupby('route').apply(lambda g: mean_absolute_error(g.actual, g.predicted)).rename('mae').reset_index()
by_daytype = {str(k): metrics(g.actual, g.predicted) for k, g in test_results.groupby('daytype')}
test_results['demand_quartile'] = pd.qcut(test_results.actual, q=4, duplicates='drop').astype(str)
by_demand = {str(k): metrics(g.actual, g.predicted) for k, g in test_results.groupby('demand_quartile')}
print('Test baselines:', json.dumps(baseline_metrics, indent=2))
print('By day type:', json.dumps(by_daytype, indent=2))
print('By demand quartile:', json.dumps(by_demand, indent=2))
display(route_mae.sort_values('mae', ascending=False).head(15))

# Fixed-origin forecast: after 2026-04-30, feed predictions (never actual test rides) into later lags.
routes = sorted(fit_df.route.astype(str).unique())
history = {(str(r.route), pd.Timestamp(r.date).normalize()): float(r.rides) for r in fit_df[['route', 'date', 'rides']].itertuples(index=False)}
daytype_by_date = data.groupby('date').daytype.agg(lambda values: values.mode().iloc[0]).to_dict()
fixed_forecasts, fixed_last_week, fixed_seasonal = [], {}, {}
for target_date in pd.date_range(test_start, test_end, freq='D'):
    target_daytype = daytype_by_date.get(target_date, 'U' if target_date.dayofweek == 6 else 'A' if target_date.dayofweek == 5 else 'W')
    future = pd.DataFrame({'route': routes, 'daytype': target_daytype, 'date': target_date})
    future['day_of_week'] = target_date.dayofweek
    future['month'] = target_date.month
    future['is_weekend'] = int(target_date.dayofweek >= 5)
    future['is_sunday_holiday'] = int(target_daytype == 'U')
    for lag_days in LAG_DAYS:
        future[f'lag_{lag_days}'] = [history.get((route, target_date - pd.Timedelta(days=lag_days))) for route in routes]
    seasonal = future[seasonal_cols]
    future['seasonal_lag_count'] = seasonal.notna().sum(axis=1)
    future['seasonal_lag_mean'] = seasonal.mean(axis=1).where(future.seasonal_lag_count >= 2)
    future_pred = np.maximum(model.predict(future[FEATURE_COLUMNS]), 0)
    for idx, route in enumerate(routes):
        key = (route, target_date)
        pred = float(future_pred[idx])
        fixed_forecasts.append({'route': route, 'date': target_date, 'predicted': pred})
        history[key] = pred
        lag7, lag1 = future.iloc[idx]['lag_7'], future.iloc[idx]['lag_1']
        past_lags = future.iloc[idx][seasonal_cols].dropna()
        fallback_value = float(fit_df.rides.median())
        fixed_last_week[key] = float(lag7 if pd.notna(lag7) else lag1 if pd.notna(lag1) else fallback_value)
        fixed_seasonal[key] = float(past_lags.mean() if len(past_lags) >= 2 else lag7 if pd.notna(lag7) else lag1 if pd.notna(lag1) else fallback_value)
fixed_forecasts = pd.DataFrame(fixed_forecasts)
fixed_truth = test_df[['route', 'date', 'rides', 'daytype']].copy()
fixed_truth['route'] = fixed_truth.route.astype(str)
fixed_results = fixed_truth.merge(fixed_forecasts, on=['route', 'date'], how='inner', validate='one_to_one')
fixed_results['last_week_same_day'] = [fixed_last_week[(r, d)] for r, d in zip(fixed_results.route, fixed_results.date)]
fixed_results['past_four_same_weekdays'] = [fixed_seasonal[(r, d)] for r, d in zip(fixed_results.route, fixed_results.date)]
fixed_origin_metrics = {
    'forecast_origin': VALIDATION_END,
    'scored_rows': len(fixed_results),
    'test_observation_rows': len(test_df),
    'scored_coverage_pct': 100 * len(fixed_results) / len(test_df),
    'daytype_assumption': 'CTA daytype is treated as known calendar metadata; verify its operational availability',
    'selected_model': metrics(fixed_results.rides, fixed_results.predicted),
    'seasonal_baselines': {
        'last_week_same_day': metrics(fixed_results.rides, fixed_results.last_week_same_day),
        'past_four_same_weekdays': metrics(fixed_results.rides, fixed_results.past_four_same_weekdays)},
    'metrics_by_daytype': {str(k): metrics(g.rides, g.predicted) for k, g in fixed_results.groupby('daytype')}
}
print('Fixed-origin test: one forecast issued at', VALIDATION_END, 'with no actual test rides used as lag inputs')
print(json.dumps(fixed_origin_metrics, indent=2))

## 6. Save artifacts

Save the fitted preprocessing/model pipeline and a JSON report. Colab will download both to your computer.

In [ ]:
report = {
    'task': 'daily route ridership regression', 'target': 'measured rides',
    'evaluation_note': 'Retrospective only; publication availability of lag observations is unverified.',
    'source_sha256': source_sha256, 'selected_model': best_name,
    'splits': {'train': {'start': HISTORY_START, 'end': TRAIN_END, 'rows': len(train_df)},
               'validation': {'start': VALIDATION_START, 'end': VALIDATION_END, 'rows': len(val_df)},
               'test': {'start': TEST_START, 'end': TEST_END, 'rows': len(test_df)}},
    'validation_model_metrics': validation_metrics,
    'test_model_metrics': metrics(test_actual, test_pred),
    'test_seasonal_baselines': baseline_metrics,
    'rolling_one_step_test_metrics': {'model': metrics(test_actual, test_pred), 'seasonal_baselines': baseline_metrics},
    'fixed_origin_test_metrics': fixed_origin_metrics,
    'test_metrics_by_daytype': by_daytype, 'test_metrics_by_demand_quartile': by_demand,
    'fallbacks': {'model': 'training median imputation with missing indicators',
                  'baselines': 'lag 7, then lag 1, then training target median'}}
model_metadata = {'pipeline': model, 'features': FEATURE_COLUMNS, 'model_name': best_name,
                  'source_sha256': source_sha256, 'artifact_version': '0.2.0',
                  'prediction_status': 'retrospective experiment; publication delay unverified'}
MODEL_PATH, REPORT_PATH = '/content/cta_ridership.joblib', '/content/model_metrics.json'
joblib.dump(model_metadata, MODEL_PATH)
with open(REPORT_PATH, 'w') as f: json.dump(report, f, indent=2)
print('Saved:', MODEL_PATH, REPORT_PATH)
from google.colab import files
files.download(MODEL_PATH)
files.download(REPORT_PATH)